# Lab Assignment 2 - Part A: Linear Regression
Please refer to the `README.pdf` for full laboratory instructions.

## Problem Statement
A dataset is included related to red and white vinho verde wine samples, from the north of Portugal. In this exercise, we look at a subset of the data and try to **predict wine's citric acid level based on other features**.

### Dataset Description
Input variables (based on physicochemical tests):
1. fixed acidity
2. volatile acidity
3. **citric acid** (TARGET - what we want to predict)
4. residual sugar
5. chlorides
6. free sulfur dioxide
7. total sulfur dioxide
8. density
9. pH
10. sulphates
11. alcohol
12. quality (score between 0 and 10)

### Your Tasks
1. **Implement linear regression** from scratch using least-squares (you may use `np.linalg.lstsq()`)
2. Start with 'alcohol' and 'density' as features. **Find a 3rd feature** that improves prediction the most
3. **Find the 4th feature**. Analyze what happens with all features
4. **Provide plots** comparing predictions vs actual values

## Setup: Load the Dataset
The data is provided through `ucimlrepo`. Install and import required packages below.

In [39]:
!pip install ucimlrepo

In [51]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
# We take 100 samples and predict the citric acid number through various features
X = wine_quality.data.features[:100]
X = X.drop(columns=['citric_acid'])
y = wine_quality.data.features[:100]['citric_acid']
print(X.keys())

Index(['fixed_acidity', 'volatile_acidity', 'residual_sugar', 'chlorides',
       'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH',
       'sulphates', 'alcohol'],
      dtype='object')


### Write and Run Your Own Code

In [ ]:
%matplotlib inline

#Library declarations
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Task 1: Implement Linear Regression
# Hint: You can use np.linalg.lstsq() or implement the normal equation: w = (X^T X)^{-1} X^T y

def linear_regression(X, y):
    """
    Implement linear regression using least-squares.
    
    Parameters:
    -----------
    X : numpy array of shape (n_samples, n_features)
    y : numpy array of shape (n_samples,)
    
    Returns:
    --------
    coefficients : numpy array
    """
    coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
    return coefficients


def compute_error(X, y, coefficients):
    """
    Compute the prediction error (RMSE).
    
    Returns:
    --------
    error : float (RMSE)
    """
    predictions = X @ coefficients
    n = len(y)
    rmse = np.sqrt(np.sum((y - predictions) ** 2) / n)
    return rmse

## Task 2: Start with Two Features
Use 'alcohol' and 'density' as initial features. Train your model and compute the error.

In [ ]:
# Create feature matrix with 'alcohol' and 'density'
X_2features = np.vstack((X['alcohol'], X['density'])).T

# Train model and compute error
model_2 = linear_regression(X_2features, y)
error_2 = compute_error(X_2features, y, model_2)
print(f"Error with 2 features (alcohol, density): {error_2:.4f}")

## Task 3: Find the 3rd Feature
Try adding each remaining feature one at a time. Which one improves the model the most?

**Hint**: You might want to look at correlations between features.


In [ ]:
# Try each remaining feature as the 3rd feature and find the one that reduces error the most
best_3rd_error = float('inf')
best_3rd_feature = None

for key in X.keys():
    if key not in ['alcohol', 'density']:
        X_3 = np.vstack((X['alcohol'], X['density'], X[key])).T
        model_3 = linear_regression(X_3, y)
        error_3 = compute_error(X_3, y, model_3)
        print(f"Error with 3rd feature ({key}): {error_3:.4f}")
        if error_3 < best_3rd_error:
            best_3rd_error = error_3
            best_3rd_feature = key

print(f"\nBest 3rd feature: {best_3rd_feature} with RMSE: {best_3rd_error:.4f}")

# Build the 3-feature model with the best choice
X_3features = np.vstack((X['alcohol'], X['density'], X[best_3rd_feature])).T
model_3 = linear_regression(X_3features, y)

## Task 4: Find the 4th Feature and Full Model
Continue the analysis. What is the best 4th feature? What happens when you use all features?


In [ ]:
# Try each remaining feature as the 4th feature
best_4th_error = float('inf')
best_4th_feature = None

for key in X.keys():
    if key not in ['alcohol', 'density', best_3rd_feature]:
        X_4 = np.vstack((X['alcohol'], X['density'], X[best_3rd_feature], X[key])).T
        model_4 = linear_regression(X_4, y)
        error_4 = compute_error(X_4, y, model_4)
        print(f"Error with 4th feature ({key}): {error_4:.4f}")
        if error_4 < best_4th_error:
            best_4th_error = error_4
            best_4th_feature = key

print(f"\nBest 4th feature: {best_4th_feature} with RMSE: {best_4th_error:.4f}")

# Build the 4-feature model
X_4features = np.vstack((X['alcohol'], X['density'], X[best_3rd_feature], X[best_4th_feature])).T
model_4 = linear_regression(X_4features, y)

# Train full model with all features
X_all = np.vstack([X[key] for key in X.keys()]).T
model_full = linear_regression(X_all, y)
error_full = compute_error(X_all, y, model_full)
print(f"\nError with all features: {error_full:.4f}")

## Task 5: Visualization
Create plots comparing model predictions vs actual values for different feature combinations.


In [ ]:
# Predictions vs actual for all models
plt.figure(figsize=(12, 6))
plt.plot(y.values, label='Actual', color='gray', linewidth=2)
plt.plot(X_2features @ model_2, label=f'2 features (RMSE={error_2:.4f})')
plt.plot(X_3features @ model_3, label=f'3 features (RMSE={best_3rd_error:.4f})')
plt.plot(X_4features @ model_4, label=f'4 features (RMSE={best_4th_error:.4f})')
plt.plot(X_all @ model_full, label=f'All features (RMSE={error_full:.4f})', linestyle='--')
plt.legend()
plt.xlabel('Sample Index')
plt.ylabel('Citric Acid')
plt.title('Linear Regression: Predictions vs Actual')
plt.show()

# Scatter plot: predicted vs actual for best 4-feature model
plt.figure(figsize=(6, 6))
preds_4 = X_4features @ model_4
plt.scatter(y.values, preds_4, alpha=0.6)
lims = [min(y.min(), preds_4.min()), max(y.max(), preds_4.max())]
plt.plot(lims, lims, 'k--', label='Ideal')
plt.xlabel('Actual Citric Acid')
plt.ylabel('Predicted Citric Acid')
plt.title('Predicted vs Actual (4-Feature Model)')
plt.legend()
plt.tight_layout()
plt.show()

## Summary and Discussion

### Results Table
| Model | Features | RMSE |
|-------|----------|------|
| Model 1 | alcohol, density | 0.1771 |
| Model 2 | alcohol, density, volatile_acidity | 0.1363 |
| Model 3 | alcohol, density, volatile_acidity, fixed_acidity | 0.1242 |
| Full Model | all features | 0.1062 |

### Discussion

- **Which features are most important for predicting citric acid?** Volatile acidity provided the largest single improvement when added as a 3rd feature, reducing RMSE from 0.1771 to 0.1363. Fixed acidity was the best 4th feature, further reducing error to 0.1242. This makes sense chemically since volatile acidity and fixed acidity are both measures of acid content in wine and are closely related to citric acid levels.

- **Does adding more features always improve the model?** On the training set, adding more features generally decreased RMSE (the full model with all 10 features achieved the lowest error of 0.1062). However, the degree of improvement varies significantly -- some features like free_sulfur_dioxide barely helped at all. In practice, with only 100 samples and many features, overfitting becomes a concern, so more features on training data does not guarantee better generalization on unseen data.

- **What did I learn from this exercise?** I learned that greedy forward feature selection can identify the most informative features for a regression model. Not all features contribute equally -- a small subset can capture most of the predictive power. I also observed that the improvement from each additional feature tends to diminish, suggesting that the first few well-chosen features carry the majority of the useful signal.